In [ ]:
import sys; sys.path.append('..')
import utils

import MeshFEM, mesh, sparse_matrices, benchmark, field_sampler, mesh_utilities, inflation, fd_validation, sheet_optimizer, opt_config
from tri_mesh_viewer import TriMeshViewer
import numpy as np, importlib
import sheet_meshing, remeshing_utils
import filters
from matplotlib import pyplot as plt

In [ ]:
sheet_opt = sheet_optimizer.load('data/sheet_opt.pkl.gz')
rso = sheet_opt.rso
isheet = rso.sheet()

### Test the raw derivatives with respect to deformed and rest vertex positions

In [ ]:
class CPWrapper:
    def __init__(self, cpenalty, useRestVars = False):
        self.sheet = cpenalty.sheet()
        self.cpenalty = cpenalty
        self.useRestVars = useRestVars
    def energy(self): return self.cpenalty.J()
    def gradient(self):
        return self.cpenalty.dJ_dX().ravel() if self.useRestVars else self.cpenalty.dJ_dx()
    def numVars(self): return self.sheet.mesh().numVertices() * 2          if self.useRestVars else self.sheet.numVars()
    def getVars(self): return self.sheet.mesh().vertices()[:, 0:2].ravel() if self.useRestVars else self.sheet.getVars()
    def setVars(self, v):
        if self.useRestVars:
            self.sheet.setRestVertexPositions(v.reshape(-1, 2))
        else:
            self.sheet.setVars(v)

In [ ]:
cp = CPWrapper(rso.compressionPenalty())

In [ ]:
# Test use of a custom modulation function. WARNING: these cannot be pickled :(
cp.cpenalty.modulation = inflation.CPModulationCustom()
cp.cpenalty.modulation. j_func = lambda x: x*x
cp.cpenalty.modulation.dj_func = lambda x: 2.0 * x

In [ ]:
cp.cpenalty.modulation = inflation.CPModulationTanh()

In [ ]:
cp.cpenalty.modulation = inflation.CPModulationPthRoot()
cp.cpenalty.modulation.set_p(4)

In [ ]:
cp.cpenalty.modulation.j(1000)

In [ ]:
cp.cpenalty.modulation.dj(0)

In [ ]:
class ModWrapper:
    def __init__(self, mod):
        self.mod = mod
        self.x = 0
    def setVars(self, val): self.x = val[0]
    def numVars(self): return 1
    def getVars(self): return np.array([self.x])
    def energy(self): return self.mod.j(self.x)
    def gradient(self): return np.array([self.mod.dj(self.x)])

In [ ]:
fd_validation.gradConvergencePlot(ModWrapper(cp.cpenalty.modulation))

In [ ]:
cp.gradient()

In [ ]:
cp.useRestVars = True
benchmark.reset()
fd_validation.gradConvergencePlot(cp)
benchmark.report()

In [ ]:
cp.useRestVars = False
benchmark.reset()
fd_validation.gradConvergencePlot(cp)
benchmark.report()

### Test the `CompressionPenalty` term's full gradient

In [ ]:
rso.compressionPenalty().modulation = inflation.CPModulationTanh()
rso.compressionPenaltyWeight = 1.0

In [ ]:
fd_validation.gradConvergencePlot(rso, epsilons=np.logspace(-9, -5, 20), customArgs={'etype': rso.EnergyType.CompressionPenalty})

In [ ]:
fd_validation.validateGrad(rso, fd_eps=1e-9, customArgs={'etype': rso.EnergyType.CompressionPenalty})

In [ ]:
rso.compressionPenalty()